# commodity-etrading-stack: results walkthrough

Loads `results/*.json` produced by `scripts/run_all.sh` and shows the figures and the headline numbers. Run the pipeline first (or `scripts/run_all.sh --skip-download` on the committed derived data).

In [ ]:
import json, os, glob
import pandas as pd
from IPython.display import Image, display, Markdown
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
load = lambda n: json.load(open(f'results/{n}', encoding='utf-8')) if os.path.exists(f'results/{n}') else None
ROOTS = ['CL', 'NG']

## 1. Data: what a free history of the curve looks like

In [ ]:
pd.DataFrame({r: load(f'datacheck_{r.lower()}.json') for r in ROOTS}).T

## 2. Curve construction: seasonal NSS vs P-spline vs seasonal kernel ridge

In [ ]:
for r in ROOTS:
    c = json.load(open(sorted(glob.glob(f'results/curve_{r.lower()}_*.json'))[-1]))
    print(r, c['date'], c['n_points'], 'contracts, gap', c['gap_months'])
    display(pd.DataFrame({m: {k: v for k, v in j.items() if not isinstance(v, (list, dict))} for m, j in c['methods'].items()}).T.round(4))
display(Image('results/figures/curve.png'))

In [ ]:
for r in ROOTS:
    h = load(f'curvehist_{r.lower()}.json')
    if h:
        print(r, h['from'], '..', h['to'], h['n_days'], 'days; selected on the first', h['n_train'])
        display(pd.DataFrame(h['selected']).T)
        display(pd.DataFrame(h['hedge_test']['methods']).T.round(4))
display(Image('results/figures/curvehist.png'))

## 3. Factor risk

In [ ]:
for r in ROOTS:
    f = load(f'factors_{r.lower()}.json')
    if f:
        rec = f['recent']; print(r, rec['from'], '..', rec['to'], rec['n_days'], 'days; explained', [round(e, 3) for e in rec['explained']])
        display(pd.DataFrame(rec['loadings'], columns=rec['months'], index=['level', 'slope', 'curvature']).round(3))
display(Image('results/figures/factors.png'))

## 4. Price impact from hourly bars

In [ ]:
for r in ROOTS:
    d = load(f'impact_{r.lower()}.json')
    if d:
        display(pd.DataFrame([{k: v for k, v in w.items() if not isinstance(v, (list, dict))} for w in d['windows']]).round(3))
display(Image('results/figures/impact.png'))

## 5. Hedging swap books at equal residual risk

In [ ]:
for r in ROOTS:
    d = load(f'hedge_{r.lower()}.json')
    if d:
        display(pd.DataFrame([{k: v for k, v in c.items() if not isinstance(v, (list, dict))} for c in d['cells']]).round(2))
    t = load(f'hedge_test_{r.lower()}.json')
    if t:
        print(r, 'EIA panel: learned/curve std ratio', t['learned_over_curve_std_ratio'], 'curve/front', t['curve_over_front_std_ratio'])
display(Image('results/figures/hedge.png'))

## 6. RFQ simulator: quoters x hedgers x risk aversion

In [ ]:
for r in ROOTS:
    d = load(f'sim_{r.lower()}.json')
    if d:
        n = d['days']
        rows = [{'hedger': c['hedger'], 'quoter': c['quoter'], 'gamma': c['gamma'], 'pnl_per_day_k': c['pnl'] / n / 1e3, 'ci_lo': c['pnl_ci95'][0] / n / 1e3, 'ci_hi': c['pnl_ci95'][1] / n / 1e3, 'std5_k': c['pnl_5min_std'] / 1e3, 'pnl_per_var_e3': 1e3 * c['pnl_per_var'], 'adverse_k': c['decomposition']['adverse'] / n / 1e3, 'hedge_k': (c['decomposition']['hedge_exec'] + c['decomposition']['hedge_impact']) / n / 1e3, 'internalisation': c['internalisation_ratio'], 'kills': c['risk']['n_kills']} for c in d['cells']]
        display(pd.DataFrame(rows).round(2))
display(Image('results/figures/sim.png'))
display(Image('results/figures/policy.png'))

In [ ]:
for r in ROOTS:
    for v in ('tox', 'anticipate', 'impact_half', 'impact_double'):
        e = load(f'sim_{v}_{r.lower()}.json')
        if e:
            print(r, v); display(pd.DataFrame([{'hedger': c['hedger'], 'quoter': c['quoter'], 'pnl_per_day_k': c['pnl'] / e['days'] / 1e3, 'pnl_per_var_e3': 1e3 * c['pnl_per_var'], 'adverse_k': c['decomposition']['adverse'] / e['days'] / 1e3, 'externalised': c['n_externalised'], 'internalisation': c['internalisation_ratio']} for c in e['cells']]).round(2))

## 7. Replay of the real 2024-2026 days with EIA release jumps

In [ ]:
for r in ROOTS:
    d = load(f'events_{r.lower()}.json')
    if d:
        n = d['n_days']; print(r, d['from'], '..', d['to'], n, 'days,', d['n_events'], 'release days')
        display(pd.DataFrame([{'quoter': c['quoter'], 'pnl_per_day_k': c['pnl'] / n / 1e3, 'ci_lo': c['pnl_ci95'][0] / n / 1e3, 'ci_hi': c['pnl_ci95'][1] / n / 1e3, 'pnl_per_var_e3': 1e3 * c['pnl_per_var'], 'adverse_k': c['decomposition']['adverse'] / n / 1e3, 'fund_hit': c['hit_ratio_by_tier'][2]} for c in d['cells']]).round(2))
display(Image('results/figures/events.png'))

## 8. Release-day study: jump share, inventory-surprise nowcast, hedging pressure

In [ ]:
pd.DataFrame({r: {k: v for k, v in (load(f'events_study_{r.lower()}.json') or {}).items() if not isinstance(v, dict)} for r in ROOTS}).T

In [ ]:
for r in ROOTS:
    e = load(f'events_study_{r.lower()}.json')
    if e: display(pd.DataFrame({'surprise_nowcast': e['surprise_nowcast'], 'hedging_pressure': e['hedging_pressure']}).T.round(3))
display(Image('results/figures/events_study.png'))

## 9. Tests

In [ ]:
print(open('results/tests.txt', encoding='utf-8').read() if os.path.exists('results/tests.txt') else 'run the pipeline first')